# Dollar Volume Exhaustion / Continuation — Backtest Riguroso

**Objetivo:** Determinar estadísticamente si el ratio de Dollar Volume genera un edge explotable en small caps después de un día de momentum (Day T).

**Hipótesis central:**
- Si T+1/T+2 aún NO han alcanzado el DV del Day T → mayor probabilidad de **continuación** (long)
- Si el DV supera al del Day T → señal de **agotamiento** → reversión o rebote

**Approach:** Escéptico y basado en datos. No asumimos que el edge existe — lo probamos o refutamos.

---

| Parámetro | Valor |
|---|---|
| Universo | Top Gainers de finviz-dashboard (Day T candidates) |
| Datos de precios | `market_bars` table (1-min OHLCV de IBKR) |
| Fallback | TWS ib_insync si `USE_TWS=True` |
| Granularidad | 1 minuto |
| Slippage | 0.05% por lado |
| Comisión | $0.005/acción |
| Shares por trade | 100 (para P&L en $) |

## 0. Imports y Configuración

In [ ]:
import sqlite3, os, re
from collections import defaultdict
from datetime import date

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.figsize': (14, 6), 'font.size': 11, 'axes.titlesize': 12})

# ─── PATHS ────────────────────────────────────────────────────────────────────
FINVIZ_DB = os.path.expanduser(
    '~/Library/Application Support/finviz-dashboard/finviz_snapshots.db'
)

# ─── UNIVERSE FILTERS (mirror finviz_to_proactive.py) ─────────────────────────
U_MIN_APPEARANCES = 3      # min snapshots en Top Gainers el día T
U_MIN_CHANGE_PCT  = 15.0   # % mínimo de subida en Day T
U_MIN_PRICE       = 0.80   # precio mínimo (evitar sub-penny)
U_MAX_PRICE       = 50.0   # precio máximo (small cap proxy)

# ─── BACKTEST PARAMETERS ──────────────────────────────────────────────────────
SLIPPAGE_PCT         = 0.0005   # 0.05% por lado
COMMISSION_PER_SHARE = 0.005    # IBKR tiered
SHARES               = 100      # tamaño fijo para P&L en $
STOP_ATR_MULT        = 1.5      # stop = ATR(14) x mult
TARGET_R             = 2.0      # target = 2R
MAX_HOLD_BARS        = 78       # ~máx 1h15min (barras de 1-min)

# ─── DV RATIO BUCKETS ─────────────────────────────────────────────────────────
RATIO_BINS   = [0, 0.3, 0.6, 1.0, 1.5, np.inf]
RATIO_LABELS = ['0–0.3x', '0.3–0.6x', '0.6–1.0x', '1.0–1.5x', '>1.5x']

# ─── MARKET HOURS (ET, naive) ─────────────────────────────────────────────────
MKT_OPEN  = pd.Timestamp('1900-01-01 09:30:00')
MKT_CLOSE = pd.Timestamp('1900-01-01 16:00:00')

# ─── TWS (opcional para descargar barras faltantes) ───────────────────────────
USE_TWS       = False   # cambiar a True si TWS está corriendo
TWS_HOST      = '127.0.0.1'
TWS_PORT      = 7497
TWS_CLIENT_ID = 5055

print(f'DB: {FINVIZ_DB}')
print(f'DB existe: {os.path.exists(FINVIZ_DB)}')

## 1. Universo Day T — carga desde finviz_snapshots.db

Los Day T candidates son tickers que aparecieron **persistentemente** (≥3 snapshots) en la
categoría *Top Gainers* de finviz-dashboard con un cambio de ≥15%. Esta es exactamente la
misma lógica de `finviz_to_proactive.py`.

In [ ]:
def _parse_pct(v) -> float:
    try: return float(re.sub(r'[%+]', '', str(v)).strip())
    except: return 0.0

def _parse_price(v) -> float:
    try: return float(re.sub(r'[$,]', '', str(v)).strip())
    except: return 0.0

def load_day_t_candidates(db: str) -> pd.DataFrame:
    """Retorna DataFrame con los Day T candidates filtrados."""
    if not os.path.exists(db):
        raise FileNotFoundError(f'DB no encontrada: {db}')

    with sqlite3.connect(db) as conn:
        rows = conn.execute("""
            SELECT ticker, price, change_pct, volume, timestamp
            FROM snapshots WHERE category = 'Top Gainers'
            ORDER BY ticker, timestamp
        """).fetchall()

    by_td = defaultdict(list)
    for ticker, price, chg, vol, ts in rows:
        by_td[(ticker, ts[:10])].append({'price': price, 'chg': chg, 'ts': ts})

    out = []
    for (ticker, day), snaps in by_td.items():
        n = len(set(s['ts'] for s in snaps))
        if n < U_MIN_APPEARANCES: continue
        prices  = [_parse_price(s['price']) for s in snaps]
        changes = [_parse_pct(s['chg'])     for s in snaps]
        max_chg = max(changes)
        lp      = prices[-1]
        if max_chg < U_MIN_CHANGE_PCT: continue
        if not (U_MIN_PRICE <= lp <= U_MAX_PRICE): continue
        out.append({
            'ticker': ticker, 'day_t': day,
            'appearances': n, 'max_change_pct': max_chg,
            'last_price': lp,
            'first_seen': snaps[0]['ts'], 'last_seen': snaps[-1]['ts'],
        })

    df = pd.DataFrame(out)
    if df.empty:
        return df
    df['day_t'] = pd.to_datetime(df['day_t'])
    return df.sort_values(['day_t', 'max_change_pct'], ascending=[True, False]).reset_index(drop=True)


candidates = load_day_t_candidates(FINVIZ_DB)
print(f'Day T candidates: {len(candidates)} registros, {candidates["ticker"].nunique()} tickers únicos')
if not candidates.empty:
    print(f'Rango fechas: {candidates["day_t"].min().date()} → {candidates["day_t"].max().date()}')
    print(f'\nDistribución de max_change_pct:')
    print(candidates['max_change_pct'].describe().round(1))
    print()
    display(candidates.tail(10))

## 2. Barras 1-min — market_bars table (+ fallback TWS)

In [ ]:
def load_bars(db: str, tickers: list, date_strs: list) -> pd.DataFrame:
    """Carga barras 1-min de market_bars para los tickers/fechas dados."""
    if not os.path.exists(db):
        return pd.DataFrame()
    with sqlite3.connect(db) as conn:
        tables = {r[0] for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'").fetchall()}
        if 'market_bars' not in tables:
            print('WARN: tabla market_bars no existe.')
            return pd.DataFrame()
        tp  = ','.join('?' * len(tickers))
        dp  = ' OR '.join('bar_time LIKE ?' for _ in date_strs)
        sql = f"SELECT ticker,bar_time,open,high,low,close,volume FROM market_bars WHERE ticker IN ({tp}) AND ({dp}) ORDER BY ticker,bar_time"
        rows = conn.execute(sql, tickers + [f'{d}%' for d in date_strs]).fetchall()

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows, columns=['ticker','ts','open','high','low','close','volume'])
    df['ts'] = pd.to_datetime(df['ts'].str[:19])
    df['date'] = df['ts'].dt.date
    for c in ['open','high','low','close','volume']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Solo horario de mercado
    t = df['ts']
    rth = ((t.dt.hour > 9) | ((t.dt.hour == 9) & (t.dt.minute >= 30))) & (t.dt.hour < 16)
    return df[rth].dropna(subset=['close','volume']).reset_index(drop=True)


def fetch_tws(ticker: str, date_str: str, db: str) -> pd.DataFrame:
    """Descarga barras de TWS y las persiste en market_bars."""
    if not USE_TWS:
        return pd.DataFrame()
    try:
        from ib_insync import IB, Stock, util
        ib = IB()
        ib.connect(TWS_HOST, TWS_PORT, clientId=TWS_CLIENT_ID)
        ct  = Stock(ticker, 'SMART', 'USD')
        ib.qualifyContracts(ct)
        raw = ib.reqHistoricalData(ct, endDateTime=f'{date_str} 16:00:00 US/Eastern',
                                   durationStr='1 D', barSizeSetting='1 min',
                                   whatToShow='TRADES', useRTH=True, formatDate=1)
        ib.disconnect()
        if not raw:
            return pd.DataFrame()
        df = util.df(raw).rename(columns={'date':'ts'})
        df['ticker'] = ticker
        df['ts'] = pd.to_datetime(df['ts'])
        df['date'] = df['ts'].dt.date
        with sqlite3.connect(db) as conn:
            conn.execute("""CREATE TABLE IF NOT EXISTS market_bars(
                ticker TEXT, bar_time TEXT, open REAL, high REAL,
                low REAL, close REAL, volume INTEGER,
                PRIMARY KEY(ticker,bar_time))""");
            for _, r in df.iterrows():
                conn.execute("INSERT OR IGNORE INTO market_bars VALUES(?,?,?,?,?,?,?)",
                             (ticker, str(r['ts'])[:19], r['open'], r['high'],
                              r['low'], r['close'], int(r['volume'])))
            conn.commit()
        return df[['ticker','ts','date','open','high','low','close','volume']]
    except Exception as e:
        print(f'  TWS error {ticker} {date_str}: {e}')
        return pd.DataFrame()


# Construir lista de fechas necesarias (Day T, T+1, T+2)
if candidates.empty:
    bars = pd.DataFrame()
else:
    needed_tickers = candidates['ticker'].unique().tolist()
    needed_dates   = set()
    for dt in candidates['day_t']:
        for i in range(3):
            needed_dates.add((dt + pd.offsets.BDay(i)).strftime('%Y-%m-%d'))
    needed_dates = sorted(needed_dates)

    bars = load_bars(FINVIZ_DB, needed_tickers, needed_dates)
    print(f'Barras en DB: {len(bars):,}')

    if USE_TWS:
        existing = set(zip(bars['ticker'].astype(str), bars['date'].astype(str)))
        missing  = [(t, d) for t in needed_tickers for d in needed_dates
                    if (t, d) not in existing]
        print(f'Faltantes: {len(missing)} → descargando de TWS...')
        extra = [fetch_tws(t, d, FINVIZ_DB) for t, d in missing[:60]]
        extra = [x for x in extra if not x.empty]
        if extra:
            bars = pd.concat([bars] + extra, ignore_index=True)
    print(f'Barras totales: {len(bars):,} | Tickers: {bars["ticker"].nunique()}')

## 3. Feature Engineering

Calculamos por barra: VWAP, DV acumulado, ATR(14), y adjuntamos el DV de referencia
(total dollar volume del Day T). El **ratio** es la variable central del estudio.

In [ ]:
def add_intraday_features(df: pd.DataFrame) -> pd.DataFrame:
    """VWAP, cumDV, ATR14, bar_index, indicadores de vela."""
    df = df.sort_values(['ticker','ts']).copy()
    g  = df.groupby(['ticker','date'])

    df['dv']      = df['close'] * df['volume']
    df['cum_dv']  = g['dv'].cumsum()
    df['cum_vol'] = g['volume'].cumsum()

    tp = (df['high'] + df['low'] + df['close']) / 3
    df['cum_tpv'] = g.apply(lambda x: (tp.loc[x.index] * x['volume']).cumsum()).reset_index(level=[0,1], drop=True)
    df['vwap']    = df['cum_tpv'] / df['cum_vol']

    pc = g['close'].shift(1)
    df['tr']   = pd.concat([
        df['high'] - df['low'],
        (df['high'] - pc).abs(),
        (df['low']  - pc).abs()
    ], axis=1).max(axis=1)
    df['atr14'] = g['tr'].transform(lambda x: x.rolling(14, min_periods=1).mean())

    df['bar_idx']   = g.cumcount()
    df['is_green']  = df['close'] >= df['open']
    df['is_red']    = df['close'] <  df['open']
    df['intraday_high'] = g['high'].cummax()
    df['intraday_low']  = g['low'].cummin()

    # Hora decimal ET
    df['hour_dec'] = df['ts'].dt.hour + df['ts'].dt.minute / 60

    return df.drop(columns=['cum_tpv','tr'], errors='ignore')


def attach_day_t_reference(bars: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
    """
    Para cada barra en T+1/T+2 adjunta:
    - day_t_total_dv : DV total del Day T para ese ticker
    - day_t_high     : high intradiario máximo del Day T
    - day_t_close    : cierre del Day T
    - days_since_t   : 1 = T+1, 2 = T+2
    - dv_ratio       : cum_dv / day_t_total_dv
    """
    if bars.empty or candidates.empty:
        return bars

    # DV diario total por (ticker, date)
    daily = bars.groupby(['ticker','date']).agg(
        total_dv=('dv','sum'),
        day_high=('high','max'),
        day_close=('close','last'),
    ).reset_index()
    daily['date_ts'] = pd.to_datetime(daily['date'])

    cand = candidates.copy()
    cand['day_t_str'] = cand['day_t'].dt.strftime('%Y-%m-%d')

    # Lookup: {(ticker, day_t_date) -> {total_dv, day_high, day_close}}
    ref = {}
    for _, row in cand.iterrows():
        t, d = row['ticker'], row['day_t']
        m = daily[(daily['ticker'] == t) & (daily['date_ts'] == d)]
        if not m.empty:
            ref[(t, d)] = {
                'total_dv':  m.iloc[0]['total_dv'],
                'day_high':  m.iloc[0]['day_high'],
                'day_close': m.iloc[0]['day_close'],
            }
        else:
            # Fallback: último precio conocido * 500k shares
            ref[(t, d)] = {'total_dv': row['last_price']*500_000,
                           'day_high': row['last_price'],
                           'day_close': row['last_price']}

    bars['date_ts'] = pd.to_datetime(bars['date'])

    ref_dv_col, ref_high_col, ref_close_col, days_col = [], [], [], []

    cand_sorted = cand.sort_values('day_t')

    for _, row in bars.iterrows():
        ticker   = row['ticker']
        bar_date = row['date_ts']

        tc = cand_sorted[(cand_sorted['ticker'] == ticker) &
                         (cand_sorted['day_t'] < bar_date)]
        if tc.empty:
            ref_dv_col.append(np.nan); ref_high_col.append(np.nan)
            ref_close_col.append(np.nan); days_col.append(np.nan)
            continue

        last_t = tc.iloc[-1]['day_t']
        r      = ref.get((ticker, last_t), {})
        bd_range = pd.bdate_range(last_t, bar_date)
        days_since = len(bd_range) - 1

        ref_dv_col.append(r.get('total_dv', np.nan))
        ref_high_col.append(r.get('day_high', np.nan))
        ref_close_col.append(r.get('day_close', np.nan))
        days_col.append(days_since)

    bars['day_t_total_dv']  = ref_dv_col
    bars['day_t_high']      = ref_high_col
    bars['day_t_close']     = ref_close_col
    bars['days_since_t']    = days_col
    bars['dv_ratio']        = bars['cum_dv'] / bars['day_t_total_dv']
    bars['ratio_bucket']    = pd.cut(bars['dv_ratio'], bins=RATIO_BINS,
                                     labels=RATIO_LABELS, right=False)
    bars = bars.drop(columns=['date_ts'], errors='ignore')
    return bars


if not bars.empty:
    print('Calculando features...')
    feat = add_intraday_features(bars)
    print('Adjuntando referencia Day T...')
    feat = attach_day_t_reference(feat, candidates)

    tradeable = feat[feat['days_since_t'].between(1, 3) & feat['dv_ratio'].notna()].copy()
    print(f'\nBarras tradeables (T+1..T+3): {len(tradeable):,}')
    print(f'Tickers con datos: {tradeable["ticker"].nunique()}')
    if not tradeable.empty:
        print(f'\nEstadísticas dv_ratio:')
        print(tradeable['dv_ratio'].describe().round(3))
else:
    feat = tradeable = pd.DataFrame()
    print('Sin barras — omitir feature engineering.')

## 4. Motor de Backtest

**Regla anti-lookahead:** la señal se genera en la barra `i`. El entry es la **apertura de la barra `i+1`**.
Esto elimina el lookahead de 1 barra (el caso más frecuente de contaminación).

In [ ]:
def execute_trade(day_bars: pd.DataFrame, sig_idx: int,
                  direction: str, atr: float) -> dict | None:
    """
    Simula una operación.

    day_bars  : barras del día, reset_index(drop=True)
    sig_idx   : índice de la barra de señal (entry en la SIGUIENTE barra)
    direction : 'long' | 'short'
    atr       : ATR(14) en la barra de señal

    Retorna dict con métricas o None si no hay entrada válida.
    """
    entry_idx = sig_idx + 1
    if entry_idx >= len(day_bars) or atr <= 0 or np.isnan(atr):
        return None

    entry_row = day_bars.iloc[entry_idx]
    raw       = entry_row['open']

    if direction == 'long':
        ep  = raw * (1 + SLIPPAGE_PCT)
        sl  = ep - atr * STOP_ATR_MULT
        tgt = ep + TARGET_R * (ep - sl)
    else:
        ep  = raw * (1 - SLIPPAGE_PCT)
        sl  = ep + atr * STOP_ATR_MULT
        tgt = ep - TARGET_R * (sl - ep)

    risk = abs(ep - sl)
    if risk < 0.001 or sl <= 0 or tgt <= 0:
        return None
    if direction == 'long'  and (sl >= ep or tgt <= ep): return None
    if direction == 'short' and (sl <= ep or tgt >= ep): return None

    max_bar = min(entry_idx + MAX_HOLD_BARS, len(day_bars))
    xp, xreason, xbar = None, 'EOD', len(day_bars) - 1

    for i in range(entry_idx, max_bar):
        r = day_bars.iloc[i]
        if direction == 'long':
            if r['low']  <= sl:  xp = sl  * (1 - SLIPPAGE_PCT); xreason='STOP';   xbar=i; break
            if r['high'] >= tgt: xp = tgt * (1 - SLIPPAGE_PCT); xreason='TARGET'; xbar=i; break
        else:
            if r['high'] >= sl:  xp = sl  * (1 + SLIPPAGE_PCT); xreason='STOP';   xbar=i; break
            if r['low']  <= tgt: xp = tgt * (1 + SLIPPAGE_PCT); xreason='TARGET'; xbar=i; break

    if xp is None:
        last = day_bars.iloc[max_bar - 1]
        xp   = last['close'] * (1 - SLIPPAGE_PCT if direction=='long' else 1 + SLIPPAGE_PCT)
        xreason = 'TIME' if max_bar < len(day_bars) else 'EOD'
        xbar    = max_bar - 1

    gross  = (xp - ep) if direction=='long' else (ep - xp)
    net_ps = gross - COMMISSION_PER_SHARE * 2
    r_mult = net_ps / risk

    return dict(
        entry_price=ep, exit_price=xp, stop_price=sl, target_price=tgt,
        exit_reason=xreason, entry_bar=entry_idx, exit_bar=xbar,
        hold_bars=xbar - entry_idx,
        r_multiple=r_mult, net_pnl=net_ps * SHARES,
        risk_per_share=risk,
    )

print('execute_trade() OK')

## 5. Estrategias

### Estrategia 1 — Continuation Long

**Tesis:** Si en T+1 el DV acumulado todavía no ha alcanzado el del Day T, el mercado aún no
ha consumido el interés. Entramos en breakout del máximo intradiario.

| Condición | Valor |
|---|---|
| Día | T+1 |
| DV ratio en señal | < 0.6 |
| Precio vs VWAP | close > VWAP |
| Breakout | close > máximo intradiario previo |
| Tiempo | entre barra 5 y barra 60 |
| Dirección | Long |

In [ ]:
def strategy_continuation_long(tradeable: pd.DataFrame) -> pd.DataFrame:
    trades = []
    sub = tradeable[tradeable['days_since_t'] == 1].copy()

    for (ticker, date_), grp in sub.groupby(['ticker', 'date']):
        grp   = grp.reset_index(drop=True)
        ph    = grp['high'].shift(1)          # high de la barra anterior
        ih_prev = grp['intraday_high'].shift(1)  # máximo intradiario hasta la barra anterior

        sigs = grp[
            (grp['dv_ratio'] < 0.6) &
            (grp['close']    > grp['vwap']) &
            (grp['close']    > ih_prev) &       # breakout del máximo intradiario previo
            (grp['is_green']) &
            (grp['bar_idx'].between(5, 60))
        ]
        if sigs.empty: continue

        si  = sigs.index[0]
        res = execute_trade(grp, si, 'long', grp.iloc[si]['atr14'])
        if res is None: continue

        sr = grp.iloc[si]
        trades.append({'ticker': ticker, 'date': date_, 'strategy': 'Continuation_Long',
                       'dv_ratio': sr['dv_ratio'], 'ratio_bucket': sr['ratio_bucket'],
                       'days_since_t': 1, 'signal_bar': si,
                       'signal_hour': sr['hour_dec'], **res})

    return pd.DataFrame(trades)


# ─── Estrategia 2 — Exhaustion Short ──────────────────────────────────────────
# Tesis: DV ya supera al Day T. Precio hace lower high respecto a la apertura
# y cae por debajo del VWAP → el momentum se agotó.

def strategy_exhaustion_short(tradeable: pd.DataFrame) -> pd.DataFrame:
    trades = []
    sub = tradeable[tradeable['days_since_t'] == 1].copy()

    for (ticker, date_), grp in sub.groupby(['ticker', 'date']):
        grp   = grp.reset_index(drop=True)
        ph    = grp['high'].shift(1)

        sigs = grp[
            (grp['dv_ratio'] >= 1.0) &
            (grp['close']    <  grp['vwap']) &
            (grp['high']     <  ph) &             # lower high
            (grp['is_red']) &
            (grp['bar_idx'].between(10, 90))
        ]
        if sigs.empty: continue

        si  = sigs.index[0]
        res = execute_trade(grp, si, 'short', grp.iloc[si]['atr14'])
        if res is None: continue

        sr = grp.iloc[si]
        trades.append({'ticker': ticker, 'date': date_, 'strategy': 'Exhaustion_Short',
                       'dv_ratio': sr['dv_ratio'], 'ratio_bucket': sr['ratio_bucket'],
                       'days_since_t': 1, 'signal_bar': si,
                       'signal_hour': sr['hour_dec'], **res})

    return pd.DataFrame(trades)


# ─── Estrategia 3 — Bounce Long ───────────────────────────────────────────────
# Tesis: el Day T tuvo un agotamiento extremo (gran caída desde máximos).
# En T+1 o T+2, tras estabilización, el precio recupera el VWAP desde abajo.
# Condición de selección: caída >= 20% desde el high del Day T.

def strategy_bounce_long(tradeable: pd.DataFrame, candidates: pd.DataFrame) -> pd.DataFrame:
    trades = []
    sub = tradeable[tradeable['days_since_t'].isin([1, 2])].copy()

    for (ticker, date_), grp in sub.groupby(['ticker', 'date']):
        grp = grp.reset_index(drop=True)
        if len(grp) < 20: continue

        day_t_high  = grp.iloc[0]['day_t_high']
        day_t_close = grp.iloc[0]['day_t_close']
        open_price  = grp.iloc[0]['open']

        if pd.isna(day_t_high) or day_t_high <= 0: continue

        # Caída desde el high del Day T hasta la apertura de este día
        drawdown_from_high = (day_t_high - open_price) / day_t_high
        if drawdown_from_high < 0.20: continue  # mínimo -20% desde high

        # Gap-down confirmado: open < day_t_close * 0.95
        if open_price >= day_t_close * 0.95: continue

        # Señal: primer cruce de VWAP desde abajo (close cruza de < a >)
        prev_close = grp['close'].shift(1)
        prev_vwap  = grp['vwap'].shift(1)
        sigs = grp[
            (grp['close']  > grp['vwap']) &
            (prev_close   <= prev_vwap) &
            (grp['bar_idx'].between(10, 120)) &
            (grp['is_green'])
        ]
        if sigs.empty: continue

        si  = sigs.index[0]
        res = execute_trade(grp, si, 'long', grp.iloc[si]['atr14'])
        if res is None: continue

        sr = grp.iloc[si]
        trades.append({'ticker': ticker, 'date': date_, 'strategy': 'Bounce_Long',
                       'dv_ratio': sr['dv_ratio'], 'ratio_bucket': sr['ratio_bucket'],
                       'days_since_t': grp.iloc[0]['days_since_t'],
                       'drawdown_from_high': drawdown_from_high,
                       'signal_bar': si, 'signal_hour': sr['hour_dec'], **res})

    return pd.DataFrame(trades)


# ─── Ejecutar ─────────────────────────────────────────────────────────────────
if not tradeable.empty:
    print('Ejecutando estrategias...')
    df1 = strategy_continuation_long(tradeable)
    df2 = strategy_exhaustion_short(tradeable)
    df3 = strategy_bounce_long(tradeable, candidates)
    all_trades = pd.concat([df1, df2, df3], ignore_index=True)
    print(f'  Continuation Long : {len(df1)} trades')
    print(f'  Exhaustion Short  : {len(df2)} trades')
    print(f'  Bounce Long       : {len(df3)} trades')
    print(f'  TOTAL             : {len(all_trades)} trades')
else:
    all_trades = df1 = df2 = df3 = pd.DataFrame()
    print('Sin datos — no se ejecutaron estrategias.')

## 6. Métricas por Estrategia

In [ ]:
def compute_metrics(trades: pd.DataFrame) -> pd.DataFrame:
    if trades.empty: return pd.DataFrame()
    rows = []
    for strat, g in trades.groupby('strategy'):
        wins   = g[g['r_multiple'] > 0]['r_multiple']
        losses = g[g['r_multiple'] <= 0]['r_multiple']
        n      = len(g)
        wr     = len(wins) / n
        avg_r  = g['r_multiple'].mean()
        std_r  = g['r_multiple'].std(ddof=1)
        pf     = (wins.sum() / abs(losses.sum())) if losses.sum() != 0 else np.inf
        cum    = g.sort_values('date')['net_pnl'].cumsum()
        mdd    = (cum - cum.cummax()).min()
        sharpe = avg_r / std_r * np.sqrt(252) if std_r > 0 else 0
        t_stat, pv = stats.ttest_1samp(g['r_multiple'], 0)
        rows.append({
            'Strategy':      strat,
            'N':             n,
            'Win Rate':      f'{wr:.1%}',
            'Avg R':         f'{avg_r:+.3f}',
            'Median R':      f"{g['r_multiple'].median():+.3f}",
            'Std R':         f'{std_r:.3f}',
            'Avg Win':       f"{wins.mean():+.3f}"  if len(wins)   else 'N/A',
            'Avg Loss':      f"{losses.mean():+.3f}" if len(losses) else 'N/A',
            'Profit Factor': f'{pf:.2f}',
            'Expectancy':    f'{avg_r:+.3f}R',
            'Total P&L $':   f"${cum.iloc[-1]:+,.0f}",
            'Max DD $':      f'${mdd:,.0f}',
            'Sharpe (ann)':  f'{sharpe:.2f}',
            'p-value':       f'{pv:.3f}',
            'Significant?':  '*** YES' if pv < 0.01 else ('* yes' if pv < 0.05 else 'NO'),
        })
    return pd.DataFrame(rows).set_index('Strategy')


if not all_trades.empty:
    m = compute_metrics(all_trades)
    print('=== MÉTRICAS POR ESTRATEGIA ===')
    display(m.T)
else:
    print('Sin trades — sin métricas.')
    print('\nVerifica:')
    print(' 1. ¿Hay datos en market_bars? → ver Cell de Diagnóstico (última celda)')
    print(' 2. ¿Los filtros son correctos? Prueba bajar U_MIN_CHANGE_PCT o U_MIN_APPEARANCES')
    print(' 3. ¿USE_TWS=True si quieres descargar barras de TWS?')

## 7. Análisis por DV Ratio Bucket

In [ ]:
if not all_trades.empty:
    bk = all_trades.groupby(['strategy','ratio_bucket']).agg(
        n=('r_multiple','count'),
        win_rate=('r_multiple', lambda x: (x>0).mean()),
        avg_r=('r_multiple','mean'),
        med_r=('r_multiple','median'),
        std_r=('r_multiple','std'),
        total_pnl=('net_pnl','sum'),
    ).reset_index()

    display(bk.round(3))

    strats = all_trades['strategy'].unique()
    fig, axes = plt.subplots(2, len(strats), figsize=(6*len(strats), 9))
    if len(strats) == 1: axes = axes.reshape(2,1)

    for j, strat in enumerate(strats):
        s = bk[bk['strategy']==strat]
        buckets = s['ratio_bucket'].astype(str)

        # Avg R
        ax = axes[0, j]
        cols = ['#2ecc71' if v > 0 else '#e74c3c' for v in s['avg_r']]
        bars = ax.bar(buckets, s['avg_r'], color=cols, alpha=0.8, edgecolor='gray')
        ax.axhline(0, color='k', lw=0.8)
        ax.set_title(f'{strat}\nAvg R por DV Ratio Bucket')
        ax.set_ylabel('Avg R')
        ax.tick_params(axis='x', rotation=30)
        for b, n in zip(bars, s['n']):
            ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                    f'n={int(n)}', ha='center', va='bottom', fontsize=9)

        # Win Rate
        ax = axes[1, j]
        cols = ['#2ecc71' if v >= 0.5 else '#e74c3c' for v in s['win_rate']]
        ax.bar(buckets, s['win_rate'], color=cols, alpha=0.8, edgecolor='gray')
        ax.axhline(0.5, color='k', lw=0.8, ls='--', label='50%')
        ax.set_title(f'{strat}\nWin Rate por DV Ratio Bucket')
        ax.set_ylabel('Win Rate')
        ax.set_ylim(0, 1)
        ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()

## 8. Equity Curves y Distribución de Retornos

In [ ]:
if not all_trades.empty:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    # 1. Equity curves
    ax = axes[0, 0]
    colors = ['#3498db','#e74c3c','#2ecc71']
    for i, strat in enumerate(all_trades['strategy'].unique()):
        g = all_trades[all_trades['strategy']==strat].sort_values('date')
        cumul = g['net_pnl'].cumsum()
        ax.plot(range(len(g)), cumul, label=strat, lw=2, color=colors[i % len(colors)])
    ax.axhline(0, color='gray', ls='--', lw=0.8)
    ax.set_title('Equity Curve ($) por Estrategia')
    ax.set_xlabel('Trade #')
    ax.set_ylabel('P&L acumulado ($)')
    ax.legend()

    # 2. Distribución de R
    ax = axes[0, 1]
    for i, strat in enumerate(all_trades['strategy'].unique()):
        g = all_trades[all_trades['strategy']==strat]['r_multiple']
        g.hist(ax=ax, bins=25, alpha=0.55, label=strat, color=colors[i % len(colors)])
    ax.axvline(0, color='red', ls='--', lw=1.5, label='0R')
    ax.set_title('Distribución de R-múltiplos')
    ax.set_xlabel('R-múltiplo')
    ax.legend()

    # 3. Win Rate por estrategia (barras)
    ax = axes[1, 0]
    wr_s = all_trades.groupby('strategy').apply(lambda x: (x['r_multiple']>0).mean()).reset_index()
    wr_s.columns = ['strategy','wr']
    bc = ['#2ecc71' if v >= 0.55 else '#f39c12' if v >= 0.45 else '#e74c3c' for v in wr_s['wr']]
    br = ax.bar(wr_s['strategy'], wr_s['wr'], color=bc, alpha=0.85, edgecolor='gray')
    ax.axhline(0.5, color='k', ls='--', lw=0.8, label='50%')
    ax.set_title('Win Rate por Estrategia')
    ax.set_ylim(0, 1)
    ax.set_ylabel('Win Rate')
    for b, v in zip(br, wr_s['wr']):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01,
                f'{v:.1%}', ha='center', va='bottom', fontweight='bold')
    ax.legend()

    # 4. Exit reason
    ax = axes[1, 1]
    er = all_trades.groupby(['strategy','exit_reason']).size().unstack(fill_value=0)
    er.plot(kind='bar', ax=ax, alpha=0.85)
    ax.set_title('Motivo de Salida por Estrategia')
    ax.set_xlabel('')
    ax.set_ylabel('N trades')
    ax.tick_params(axis='x', rotation=15)
    ax.legend(title='Exit')

    plt.tight_layout()
    plt.show()

## 9. Análisis por Hora de Entrada

In [ ]:
if not all_trades.empty and 'signal_hour' in all_trades.columns:
    t = all_trades.dropna(subset=['signal_hour']).copy()
    t['hour_bin'] = pd.cut(
        t['signal_hour'],
        bins=[9.5, 10.0, 10.5, 11.0, 11.5, 12.0, 13.0, 14.0, 15.0, 16.0],
        labels=['9:30','10:00','10:30','11:00','11:30','12:00','13:00','14:00','15:00'],
        right=False
    )
    hs = t.groupby(['strategy','hour_bin']).agg(
        n=('r_multiple','count'),
        avg_r=('r_multiple','mean'),
        wr=('r_multiple', lambda x: (x>0).mean()),
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    colors = ['#3498db','#e74c3c','#2ecc71']

    for ax, col, title in [(axes[0],'avg_r','Avg R'), (axes[1],'wr','Win Rate')]:
        for i, strat in enumerate(all_trades['strategy'].unique()):
            sg = hs[hs['strategy']==strat]
            ax.plot(sg['hour_bin'].astype(str), sg[col],
                    marker='o', lw=2, label=strat, color=colors[i % len(colors)])
        ref = 0 if col == 'avg_r' else 0.5
        ax.axhline(ref, color='gray', ls='--', lw=0.8)
        if col == 'wr': ax.set_ylim(0, 1)
        ax.set_title(f'{title} por Hora de Entrada')
        ax.set_xlabel('Hora ET')
        ax.set_ylabel(title)
        ax.legend()
        ax.tick_params(axis='x', rotation=30)

    plt.tight_layout()
    plt.show()

## 10. Análisis por Régimen de Volatilidad

Dividimos los trades en **alta / baja volatilidad** usando el ATR relativo al precio en la señal.
Hipótesis: en alta volatilidad las estrategias de agotamiento tienen mejor edge.

In [ ]:
if not all_trades.empty and not tradeable.empty:
    # Adjuntar ATR relativo al precio (ATR%) desde la barra de señal
    atr_map = {}
    for (ticker, date_), grp in tradeable.groupby(['ticker','date']):
        grp = grp.reset_index(drop=True)
        for _, r in grp.iterrows():
            atr_map[(ticker, str(date_), int(r['bar_idx']))] = (
                r['atr14'] / r['close'] if r['close'] > 0 else np.nan
            )

    def get_atr_pct(row):
        return atr_map.get((row['ticker'], str(row['date']), int(row.get('signal_bar', 0))), np.nan)

    all_trades['atr_pct'] = all_trades.apply(get_atr_pct, axis=1)

    median_atr = all_trades['atr_pct'].median()
    all_trades['vol_regime'] = np.where(all_trades['atr_pct'] >= median_atr, 'High Vol', 'Low Vol')

    vol_stats = all_trades.groupby(['strategy','vol_regime']).agg(
        n=('r_multiple','count'),
        avg_r=('r_multiple','mean'),
        wr=('r_multiple', lambda x: (x>0).mean()),
        pnl=('net_pnl','sum'),
    ).reset_index()

    print(f'Mediana ATR%: {median_atr:.3%}')
    display(vol_stats.round(3))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, col, title in [(axes[0],'avg_r','Avg R'), (axes[1],'wr','Win Rate')]:
        pivot = vol_stats.pivot(index='strategy', columns='vol_regime', values=col)
        pivot.plot(kind='bar', ax=ax, alpha=0.85, edgecolor='gray')
        ref = 0 if col=='avg_r' else 0.5
        ax.axhline(ref, color='k', ls='--', lw=0.8)
        if col=='wr': ax.set_ylim(0,1)
        ax.set_title(f'{title} por Régimen de Volatilidad')
        ax.set_xlabel('')
        ax.tick_params(axis='x', rotation=15)
        ax.legend(title='Régimen')
    plt.tight_layout()
    plt.show()

## 11. Trades Anotados — Ejemplos Visuales

Para cada estrategia mostramos el mejor trade (mayor R) con precio, VWAP,
entry/exit y el DV ratio a lo largo del día.

In [ ]:
def plot_annotated_trade(trade_row: pd.Series, feat_df: pd.DataFrame):
    """Dibuja un trade anotado con precio, VWAP, volumen y DV ratio."""
    ticker   = trade_row['ticker']
    date_    = trade_row['date']
    strategy = trade_row['strategy']
    entry_b  = int(trade_row.get('entry_bar', 1))
    exit_b   = int(trade_row.get('exit_bar',  1))
    ep       = trade_row['entry_price']
    xp       = trade_row['exit_price']
    sl       = trade_row['stop_price']
    tgt      = trade_row['target_price']
    r        = trade_row['r_multiple']

    day_bars = feat_df[(feat_df['ticker']==ticker) & (feat_df['date']==date_)].reset_index(drop=True)
    if day_bars.empty:
        print(f'Sin barras para {ticker} {date_}')
        return

    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(14, 10),
                                         gridspec_kw={'height_ratios':[3,1,1]})
    fig.suptitle(f'{strategy} | {ticker} | {date_} | R={r:+.2f}', fontsize=13, fontweight='bold')

    xs = range(len(day_bars))
    # Candlestick simplificado (line chart de close)
    closes = day_bars['close']
    ax1.plot(xs, closes, color='#2c3e50', lw=1.2, label='Close')
    ax1.plot(xs, day_bars['vwap'], color='#8e44ad', lw=1.5, ls='--', label='VWAP', alpha=0.8)
    ax1.fill_between(xs, day_bars['intraday_high'], day_bars['intraday_low'],
                     alpha=0.06, color='blue')

    # Zonas de precio
    ax1.axhline(ep,  color='#2ecc71', lw=1.5, ls='-',  label=f'Entry  ${ep:.2f}')
    ax1.axhline(sl,  color='#e74c3c', lw=1.2, ls='--', label=f'Stop   ${sl:.2f}')
    ax1.axhline(tgt, color='#3498db', lw=1.2, ls='--', label=f'Target ${tgt:.2f}')

    # Marcadores entry/exit
    if entry_b < len(day_bars):
        ax1.axvline(entry_b, color='#2ecc71', lw=1.5, alpha=0.8)
        ax1.scatter(entry_b, ep, color='#2ecc71', s=100, zorder=5)
    if exit_b < len(day_bars):
        ax1.axvline(exit_b, color='#e74c3c', lw=1.5, alpha=0.8)
        ax1.scatter(exit_b, xp, color='#e74c3c', s=100, zorder=5,
                    marker='^' if r>0 else 'v')

    # Label de exit reason
    ax1.annotate(f"{trade_row['exit_reason']} {r:+.2f}R",
                 xy=(exit_b, xp), xytext=(exit_b+2, xp),
                 fontsize=9, color='#e74c3c',
                 arrowprops=dict(arrowstyle='->', color='#e74c3c', lw=1))

    ax1.set_ylabel('Precio')
    ax1.legend(loc='upper left', fontsize=8)
    ax1.grid(True, alpha=0.3)

    # Volumen
    vol_colors = ['#2ecc71' if g else '#e74c3c' for g in day_bars['is_green']]
    ax2.bar(xs, day_bars['volume'], color=vol_colors, alpha=0.7, width=0.9)
    ax2.set_ylabel('Volumen')
    ax2.grid(True, alpha=0.2)

    # DV Ratio
    ax3.plot(xs, day_bars['dv_ratio'], color='#f39c12', lw=1.8, label='DV Ratio')
    for thresh, col, lbl in [(0.6,'#3498db','0.6x'),(1.0,'#e74c3c','1.0x'),(1.5,'#8e44ad','1.5x')]:
        ax3.axhline(thresh, color=col, ls='--', lw=0.8, label=lbl)
    if entry_b < len(day_bars):
        r_at_entry = day_bars.iloc[entry_b]['dv_ratio']
        ax3.scatter(entry_b, r_at_entry, color='#2ecc71', s=80, zorder=5)
    ax3.set_ylabel('DV Ratio')
    ax3.set_xlabel('Barra (1-min)')
    ax3.legend(fontsize=8)
    ax3.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.show()


if not all_trades.empty and not feat.empty:
    print('=== TRADES ANOTADOS — MEJOR TRADE POR ESTRATEGIA ===\n')
    for strat in all_trades['strategy'].unique():
        sub = all_trades[all_trades['strategy']==strat]
        if sub.empty: continue
        best = sub.loc[sub['r_multiple'].idxmax()]
        print(f'--- {strat}: R={best["r_multiple"]:+.2f} | {best["ticker"]} {best["date"]} ---')
        plot_annotated_trade(best, feat)
else:
    print('Sin trades para visualizar.')

## 12. Test Estadístico de Robustez — Bootstrap

In [ ]:
def bootstrap_expectancy(r_series: pd.Series, n_boot: int = 2000, seed: int = 42) -> dict:
    """Bootstrap IC 95% para el Avg R (expectancy)."""
    rng  = np.random.default_rng(seed)
    data = r_series.dropna().values
    if len(data) < 10:
        return {'mean': np.mean(data), 'ci_lo': np.nan, 'ci_hi': np.nan}
    boots = [rng.choice(data, size=len(data), replace=True).mean() for _ in range(n_boot)]
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {'mean': np.mean(data), 'ci_lo': lo, 'ci_hi': hi}


if not all_trades.empty:
    print('=== BOOTSTRAP IC 95% — EXPECTANCY (Avg R) ===')
    rows = []
    for strat, g in all_trades.groupby('strategy'):
        res = bootstrap_expectancy(g['r_multiple'])
        rows.append({
            'Strategy': strat,
            'N': len(g),
            'Avg R': f"{res['mean']:+.3f}",
            'IC 95% Lo': f"{res['ci_lo']:+.3f}" if not np.isnan(res['ci_lo']) else 'N/A',
            'IC 95% Hi': f"{res['ci_hi']:+.3f}" if not np.isnan(res['ci_hi']) else 'N/A',
            'Edge?': 'YES' if (not np.isnan(res['ci_lo']) and res['ci_lo'] > 0) else
                     ('YES (marginal)' if (not np.isnan(res['ci_hi']) and res['ci_hi'] > 0 and res['mean'] > 0) else 'NO')
        })
    display(pd.DataFrame(rows).set_index('Strategy'))

    # Visualizar
    fig, ax = plt.subplots(figsize=(10, 4))
    strategies = [r['Strategy'] for r in rows]
    means  = [float(r['Avg R']) for r in rows]
    lo_err = [abs(float(r['Avg R']) - float(r['IC 95% Lo'])) if r['IC 95% Lo'] != 'N/A' else 0 for r in rows]
    hi_err = [abs(float(r['IC 95% Hi']) - float(r['Avg R'])) if r['IC 95% Hi'] != 'N/A' else 0 for r in rows]

    ax.bar(strategies, means, color=['#2ecc71' if m>0 else '#e74c3c' for m in means],
           alpha=0.8, edgecolor='gray', yerr=[lo_err, hi_err], capsize=8)
    ax.axhline(0, color='k', lw=1)
    ax.set_title('Avg R con Bootstrap IC 95%')
    ax.set_ylabel('Avg R')
    plt.tight_layout()
    plt.show()

## 13. Diagnóstico de Datos

In [ ]:
print('=== DIAGNÓSTICO DE COBERTURA DE DATOS ===')
print(f'DB: {FINVIZ_DB}  |  Existe: {os.path.exists(FINVIZ_DB)}')

if os.path.exists(FINVIZ_DB):
    with sqlite3.connect(FINVIZ_DB) as conn:
        tables = {r[0] for r in conn.execute(
            "SELECT name FROM sqlite_master WHERE type='table'").fetchall()}
        print(f'Tablas: {sorted(tables)}')

        if 'snapshots' in tables:
            n_snap = conn.execute("SELECT COUNT(*) FROM snapshots WHERE category='Top Gainers'").fetchone()[0]
            days_snap = conn.execute(
                "SELECT DISTINCT substr(timestamp,1,10) FROM snapshots WHERE category='Top Gainers' ORDER BY 1 DESC LIMIT 15"
            ).fetchall()
            print(f'\nSnapshots Top Gainers: {n_snap:,}')
            print(f'Últimos días: {[r[0] for r in days_snap]}')

        if 'market_bars' in tables:
            n_bars = conn.execute("SELECT COUNT(*) FROM market_bars").fetchone()[0]
            tickers_bars = conn.execute(
                "SELECT ticker, COUNT(*) n, MIN(bar_time), MAX(bar_time) FROM market_bars GROUP BY ticker ORDER BY n DESC LIMIT 20"
            ).fetchall()
            print(f'\nBarras en market_bars: {n_bars:,}')
            print('Tickers con más barras:')
            for t, n, mn, mx in tickers_bars:
                print(f'  {t:8s}  {n:5d}  {mn[:10]} → {mx[:10]}')
        else:
            print('\nATENCION: market_bars no existe.')
            print('  Para obtener datos:')
            print('  1. Activar USE_TWS=True con TWS corriendo (clientId=5055)')
            print('  2. Las barras se descargarán y persistirán automáticamente')

print()
if not candidates.empty:
    tickers_with_bars = set(bars['ticker']) if not bars.empty else set()
    covered = candidates[candidates['ticker'].isin(tickers_with_bars)]
    missing_bars_df = candidates[~candidates['ticker'].isin(tickers_with_bars)]
    print(f'Day T candidates: {len(candidates)}')
    print(f'Con barras: {len(covered)} ({len(covered)/len(candidates):.0%})')
    print(f'Sin barras: {len(missing_bars_df)} — {sorted(missing_bars_df["ticker"].unique())[:30]}')

---

## 14. Conclusiones

### ¿Existe el edge?

Interpreta los resultados con estos criterios mínimos:

| Criterio | Umbral mínimo para operar en live |
|---|---|
| N trades | ≥ 30 por estrategia |
| Win Rate (Long) | ≥ 55% |
| Win Rate (Short) | ≥ 45% (si RR ≥ 2) |
| Avg R | ≥ +0.15R |
| Profit Factor | ≥ 1.3 |
| p-value | < 0.05 |
| Bootstrap IC 95% | todo el intervalo > 0 |

### Señales de alerta (edge falso)
- N trades bajo → resultados no significativos estadísticamente
- Avg R positivo pero p > 0.05 → coincidencia, no edge
- Todos los profits vienen de 1-2 trades outliers → frágil
- Sharpe alto pero MDD muy alto → no sostenible

### Limitaciones conocidas
1. **Slippage real**: en small caps puede ser 3–10x el modelado (0.05%). El edge puede desaparecer en live.
2. **Short selling**: muchos small caps no tienen shares disponibles. Exhaustion Short puede ser teórico.
3. **Survivorship bias parcial**: solo tenemos tickers que ya ganaron en Day T.
4. **Datos limitados**: el universo depende de los días capturados por finviz-dashboard.

### Mejoras sugeridas
1. **Filtro de float**: si tienes datos de float, filtrar float < 5M (máximo volátiles)
2. **Filtro de catalizador**: solo operar si hay news en Day T (FDA, earnings, PR)
3. **Trailing stop**: en Continuation Long, mover el stop al breakeven tras 1R
4. **Time filter**: según el análisis por hora, puede que solo funcione en las primeras 2h
5. **Régimen de mercado**: en bear market (SPY < SMA20), reducir tamaño en longs

### Implementación en live (pseudocódigo)
```python
# En V5 engine — POST_BURST_T1 strategy (ya implementado)
# El DV ratio se puede calcular en tiempo real con las barras de ib_insync
# y comparar con el day_t_total_dv guardado en key_levels['burst_volume']

dv_ratio = cum_dv_today / key_levels['burst_volume']

if days_since_burst == 1 and dv_ratio < 0.6 and close > vwap:
    # Señal Continuation Long → POST_BURST_T1 trigger
    pass

if days_since_burst == 1 and dv_ratio > 1.0 and close < vwap:
    # Señal Exhaustion → no entrar long, considerar short
    pass
```